In [ ]:
import pandas as pd

df1 = pd.read_csv("../data/processed/final_features_v1.csv", index_col=0)
df2 = pd.read_csv("../data/processed/credit_risk_clean_v1.csv",index_col=0)
df3 = pd.read_csv("../data/interim/feature_engineering_v1.csv",index_col=0)

print(df1.shape)
df1.head()


In [ ]:
print(df2.shape)
df2.head()


In [ ]:
print(df3.shape)
df3.head()


In [ ]:
print(df1.info())

In [ ]:
print(df2.info())

In [ ]:
print(df3.info())

In [ ]:
df3.select_dtypes(include=["object", "string"]).columns.tolist()

In [ ]:
df4 = df3.drop(columns=['IncomePerDependent_bin',
 'TotalLatePayments_bin',
 'CreditExposure_bin',
 'DelinquencySeverityScore_bin',
 'DebtIncomeInteraction_bin'])

df4.shape

In [ ]:
for col in df3.select_dtypes(include=["object", "string"]).columns:
    print(f"\n{col}")
    print(df3[col].value_counts(dropna=False))

In [ ]:
for col in df3.select_dtypes(include="object").columns:

    df3[col] = (
        pd.Categorical(
            df3[col],
            ordered=True
        ).codes
    )

In [ ]:
df3.dtypes.value_counts()

In [ ]:
df3.select_dtypes(include="object").columns

In [ ]:
import pandas as pd
from xgboost import XGBClassifier

from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    precision_score,
    recall_score,
    f1_score
)

datasets = {
    "df1": df1,
    "df2": df2,
    "df3": df3,
    "df4": df4
}

results = []

cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

for name, df in datasets.items():

    X = df.drop(columns=["SeriousDlqin2yrs"])
    y = df["SeriousDlqin2yrs"]

    scale_pos_weight = (
        (y == 0).sum() /
        (y == 1).sum()
    )

    model = XGBClassifier(
        objective="binary:logistic",
        eval_metric="logloss",
        random_state=42,
        n_estimators=300,
        learning_rate=0.05,
        max_depth=6,
        subsample=0.8,
        colsample_bytree=0.8,
        scale_pos_weight=scale_pos_weight,
        n_jobs=-1
    )

    probs = cross_val_predict(
        model,
        X,
        y,
        cv=cv,
        method="predict_proba",
        n_jobs=-1
    )[:, 1]

    preds = (probs >= 0.5).astype(int)

    results.append({

        "Dataset": name,
        "Rows": X.shape[0],
        "Features": X.shape[1],
        "ROC-AUC": roc_auc_score(y, probs),
        "PR-AUC": average_precision_score(y, probs),
        "Precision": precision_score(y, preds),
        "Recall": recall_score(y, preds),
        "F1": f1_score(y, preds)

    })

comparison_df = (
    pd.DataFrame(results)
    .sort_values(
        by="ROC-AUC",
        ascending=False
    )
    .reset_index(drop=True)
)

display(comparison_df)

In [ ]:
import numpy as np

LOWER = 0.005      # 0.5th percentile
UPPER = 0.995      # 99.5th percentile


def trim_percentile_outliers(df, lower=LOWER, upper=UPPER):

    temp = df.copy()

    numeric_cols = temp.select_dtypes(include=np.number).columns.tolist()

    if "SeriousDlqin2yrs" in numeric_cols:
        numeric_cols.remove("SeriousDlqin2yrs")

    mask = np.ones(len(temp), dtype=bool)

    for col in numeric_cols:

        low = temp[col].quantile(lower)
        high = temp[col].quantile(upper)

        mask &= temp[col].between(low, high)

    return temp.loc[mask].reset_index(drop=True)


# Create new datasets
df1_v2 = trim_percentile_outliers(df1)
df2_v2 = trim_percentile_outliers(df2)
df3_v2 = trim_percentile_outliers(df3)
df4_v2 = trim_percentile_outliers(df4)


# Summary
summary = pd.DataFrame({
    "Dataset": ["df1_v2", "df2_v2", "df3_v2", "df4_v2"],
    "Rows": [
        len(df1_v2),
        len(df2_v2),
        len(df3_v2),
        len(df4_v2)
    ],
    "Removed": [
        len(df1) - len(df1_v2),
        len(df2) - len(df2_v2),
        len(df3) - len(df3_v2),
        len(df4) - len(df4_v2)
    ]
})

display(summary)

In [ ]:
datasets = {
    "df1_v2": df1_v2,
    "df2_v2": df2_v2,
    "df3_v2": df3_v2,
    "df4_v2": df4_v2
}


results = []

cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

for name, df in datasets.items():

    X = df.drop(columns=["SeriousDlqin2yrs"])
    y = df["SeriousDlqin2yrs"]

    scale_pos_weight = (
        (y == 0).sum() /
        (y == 1).sum()
    )

    model = XGBClassifier(
        objective="binary:logistic",
        eval_metric="logloss",
        random_state=42,
        n_estimators=300,
        learning_rate=0.05,
        max_depth=6,
        subsample=0.8,
        colsample_bytree=0.8,
        scale_pos_weight=scale_pos_weight,
        n_jobs=-1
    )

    probs = cross_val_predict(
        model,
        X,
        y,
        cv=cv,
        method="predict_proba",
        n_jobs=-1
    )[:, 1]

    preds = (probs >= 0.5).astype(int)

    results.append({

        "Dataset": name,
        "Rows": X.shape[0],
        "Features": X.shape[1],
        "ROC-AUC": roc_auc_score(y, probs),
        "PR-AUC": average_precision_score(y, probs),
        "Precision": precision_score(y, preds),
        "Recall": recall_score(y, preds),
        "F1": f1_score(y, preds)

    })

comparison_df = (
    pd.DataFrame(results)
    .sort_values(
        by="ROC-AUC",
        ascending=False
    )
    .reset_index(drop=True)
)

display(comparison_df)

### choose df3 for furthur investigation

In [ ]:
final_df = df3.copy()

X = final_df.drop(columns=["SeriousDlqin2yrs"])
y = final_df["SeriousDlqin2yrs"]

print(final_df.shape)

In [ ]:
import numpy as np
import pandas as pd
import optuna

from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier

from sklearn.model_selection import (
    StratifiedKFold,
    cross_val_score
)

from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    precision_score,
    recall_score,
    f1_score
)

In [ ]:
TARGET = "SeriousDlqin2yrs"

final_df = df3.copy()

X = final_df.drop(columns=[TARGET])
y = final_df[TARGET]

cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

scale_pos_weight = (
    (y == 0).sum() /
    (y == 1).sum()
)

print(X.shape)
print(y.value_counts())

In [ ]:
models = {

    "XGBoost": XGBClassifier(
        objective="binary:logistic",
        eval_metric="logloss",
        random_state=42,
        n_jobs=-1
    ),

    "LightGBM": LGBMClassifier(
        objective="binary",
        random_state=42,
        verbose=-1
    ),

    "CatBoost": CatBoostClassifier(
        loss_function="Logloss",
        random_state=42,
        verbose=0
    )

}

In [ ]:
param_spaces = {

    "XGBoost": {

        "n_estimators": (200,700),
        "learning_rate": (0.01,0.2,"log"),
        "max_depth": (3,10),
        "subsample": (0.6,1.0),
        "colsample_bytree": (0.6,1.0),
        "min_child_weight": (1,10),
        "gamma": (0.0,5.0),
        "reg_alpha": (1e-8,10.0,"log"),
        "reg_lambda": (1e-8,10.0,"log")

    },

    "LightGBM": {

        "n_estimators": (200,700),
        "learning_rate": (0.01,0.2,"log"),
        "num_leaves": (20,200),
        "max_depth": (3,12),
        "min_child_samples": (10,100),
        "subsample": (0.6,1.0),
        "colsample_bytree": (0.6,1.0),
        "reg_alpha": (1e-8,10.0,"log"),
        "reg_lambda": (1e-8,10.0,"log")

    },

    "CatBoost": {

        "iterations": (200,700),
        "learning_rate": (0.01,0.2,"log"),
        "depth": (4,10),
        "l2_leaf_reg": (1.0,20.0),
        "random_strength": (0.0,5.0),
        "bagging_temperature": (0.0,5.0)

    }

}

In [ ]:
def tune_model(name, model, space, n_trials=50):

    def objective(trial):

        params = {}

        for key, value in space.items():

            if len(value) == 2:

                low, high = value

                if isinstance(low, int):

                    params[key] = trial.suggest_int(
                        key,
                        low,
                        high
                    )

                else:

                    params[key] = trial.suggest_float(
                        key,
                        low,
                        high
                    )

            else:

                low, high, mode = value

                params[key] = trial.suggest_float(
                    key,
                    low,
                    high,
                    log=(mode=="log")
                )

        if name == "XGBoost":

            params.update({

                "objective":"binary:logistic",
                "eval_metric":"logloss",
                "random_state":42,
                "n_jobs":-1,
                "scale_pos_weight":scale_pos_weight

            })

            estimator = XGBClassifier(**params)

        elif name == "LightGBM":

            params.update({

                "objective":"binary",
                "random_state":42,
                "verbose":-1,
                "class_weight":"balanced"

            })

            estimator = LGBMClassifier(**params)

        else:

            params.update({

                "loss_function":"Logloss",
                "random_state":42,
                "verbose":0,
                "auto_class_weights":"Balanced"

            })

            estimator = CatBoostClassifier(**params)

        score = cross_val_score(

            estimator,
            X,
            y,
            cv=cv,
            scoring="roc_auc",
            n_jobs=-1

        ).mean()

        return score

    study = optuna.create_study(
        direction="maximize"
    )

    study.optimize(
        objective,
        n_trials=n_trials
    )

    return study

In [ ]:
import warnings
warnings.filterwarnings("ignore")

In [ ]:
studies = {}

for name in models:

    print("="*60)
    print(name)
    print("="*60)

    studies[name] = tune_model(
        name,
        models[name],
        param_spaces[name],
        n_trials=50
    )

In [ ]:
best_params = []

for name, study in studies.items():

    best_params.append({

        "Model": name,
        "ROC-AUC": study.best_value,
        "Best Parameters": study.best_params

    })

best_params = (
    pd.DataFrame(best_params)
    .sort_values(
        "ROC-AUC",
        ascending=False
    )
    .reset_index(drop=True)
)

display(best_params)

In [ ]:
from sklearn.model_selection import cross_val_predict
from sklearn.metrics import (
    accuracy_score,
    roc_auc_score,
    average_precision_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix
)
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier

tuned_models = {

    "CatBoost": CatBoostClassifier(
        **studies["CatBoost"].best_params,
        loss_function="Logloss",
        auto_class_weights="Balanced",
        random_state=42,
        verbose=0
    ),

    "XGBoost": XGBClassifier(
        **studies["XGBoost"].best_params,
        objective="binary:logistic",
        eval_metric="logloss",
        scale_pos_weight=scale_pos_weight,
        random_state=42,
        n_jobs=-1
    ),

    "LightGBM": LGBMClassifier(
        **studies["LightGBM"].best_params,
        objective="binary",
        class_weight="balanced",
        random_state=42,
        verbose=-1
    )

}

results = []

for name, model in tuned_models.items():

    probs = cross_val_predict(
        model,
        X,
        y,
        cv=cv,
        method="predict_proba",
        n_jobs=-1
    )[:, 1]

    preds = (probs >= 0.5).astype(int)

    tn, fp, fn, tp = confusion_matrix(y, preds).ravel()

    results.append({

        "Model": name,

        "Accuracy": accuracy_score(y, preds),

        "ROC-AUC": roc_auc_score(y, probs),

        "PR-AUC": average_precision_score(y, probs),

        "Precision": precision_score(y, preds),

        "Recall": recall_score(y, preds),

        "F1": f1_score(y, preds),

        "TN": tn,
        "FP": fp,
        "FN": fn,
        "TP": tp

    })

results_df = (
    pd.DataFrame(results)
    .sort_values(
        by="ROC-AUC",
        ascending=False
    )
    .reset_index(drop=True)
)

display(results_df)

### choosed catboost

In [ ]:
from sklearn.model_selection import cross_val_predict
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    precision_score,
    recall_score,
    f1_score
)

# Out-of-fold probabilities
cat_probs = cross_val_predict(
    tuned_models["CatBoost"],
    X,
    y,
    cv=cv,
    method="predict_proba",
    n_jobs=-1
)[:, 1]


results = []

for threshold in [i/100 for i in range(1,100)]:

    preds = (cat_probs >= threshold).astype(int)

    precision = precision_score(y, preds, zero_division=0)
    recall = recall_score(y, preds)
    f1 = f1_score(y, preds)

    results.append({

        "Threshold": threshold,
        "Precision": precision,
        "Recall": recall,
        "F1": f1,
        "ROC-AUC": roc_auc_score(y, cat_probs),
        "PR-AUC": average_precision_score(y, cat_probs)

    })

threshold_df = pd.DataFrame(results)

In [ ]:
candidate_thresholds = (
    threshold_df
    .query("Recall >= 0.75")
    .sort_values(
        by=["Precision", "F1"],
        ascending=False
    )
    .reset_index(drop=True)
)

display(candidate_thresholds.head(20))

In [ ]:
best_threshold = candidate_thresholds.iloc[0]

print(best_threshold)

In [ ]:
for target in [0.85, 0.80, 0.75, 0.70, 0.65]:

    best = (
        threshold_df
        .query(f"Recall >= {target}")
        .sort_values(
            by="Precision",
            ascending=False
        )
        .head(1)
    )

    print(f"\nRecall ≥ {target:.0%}")
    display(best)

In [ ]:
df3.columns

In [ ]:
from catboost import CatBoostClassifier
import pandas as pd

final_catboost = CatBoostClassifier(
    **studies["CatBoost"].best_params,
    loss_function="Logloss",
    auto_class_weights="Balanced",
    random_state=42,
    verbose=0
)

final_catboost.fit(
    X,
    y
)

In [ ]:
engineered = [
    "MonthlyIncome_log",
    "RevolvingUtilization_log",
    "DebtRatio_log",
    "IncomePerDependent",
    "IncomePerDependent_bin",
    "TotalLatePayments",
    "TotalLatePayments_bin",
    "AnyLatePayment",
    "SevereDelinquencyFlag",
    "HighDebtRatioFlag",
    "HighUtilizationFlag",
    "CreditExposure",
    "CreditExposure_bin",
    "HasDependents",
    "DelinquencySeverityScore",
    "DelinquencySeverityScore_bin",
    "AgeRiskCategory",
    "DebtIncomeInteraction",
    "DebtIncomeInteraction_bin",
    "IncomeMissingFlag"
]

importance[
    importance["Feature"].isin(engineered)
]

In [ ]:
importance = pd.DataFrame({

    "Feature": X.columns,

    "Importance": final_catboost.feature_importances_

})


importance = (
    importance
    .sort_values(
        "Importance",
        ascending=False
    )
    .reset_index(drop=True)
)

display(importance)

In [ ]:
engineered_importance = importance[
    importance["Feature"].isin(engineered)
]

display(engineered_importance)

In [ ]:
df3["SeriousDlqin2yrs"].value_counts()

In [ ]:
from imblearn.pipeline import Pipeline
from imblearn.over_sampling import SMOTE

from catboost import CatBoostClassifier

from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    precision_score,
    recall_score,
    f1_score,
    accuracy_score
)


X = df3.drop(columns=["SeriousDlqin2yrs"])
y = df3["SeriousDlqin2yrs"]


smote_catboost = Pipeline([

    (
        "smote",
        SMOTE(
            sampling_strategy=0.5,
            random_state=42,
            k_neighbors=5
        )
    ),

    (
        "model",
        CatBoostClassifier(
            iterations=567,
            learning_rate=0.0316,
            depth=6,
            loss_function="Logloss",
            random_seed=42,
            verbose=0
        )
    )

])


cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)


probs = cross_val_predict(
    smote_catboost,
    X,
    y,
    cv=cv,
    method="predict_proba",
    n_jobs=-1
)[:,1]


preds = (probs >= 0.5).astype(int)


results = {

    "Accuracy":
        accuracy_score(y,preds),

    "ROC-AUC":
        roc_auc_score(y,probs),

    "PR-AUC":
        average_precision_score(y,probs),

    "Precision":
        precision_score(y,preds),

    "Recall":
        recall_score(y,preds),

    "F1":
        f1_score(y,preds)

}


pd.Series(results)

In [ ]:
import pandas as pd
import numpy as np

from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.metrics import (
    accuracy_score,
    roc_auc_score,
    average_precision_score,
    precision_score,
    recall_score,
    f1_score
)

from imblearn.pipeline import Pipeline
from imblearn.over_sampling import SMOTE, ADASYN
from imblearn.combine import SMOTETomek

from catboost import CatBoostClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier


# ==========================
# DATA
# ==========================

X = df3.drop(columns=["SeriousDlqin2yrs"])
y = df3["SeriousDlqin2yrs"]


# ==========================
# MODELS
# ==========================

models = {

    "CatBoost": CatBoostClassifier(
        iterations=500,
        learning_rate=0.05,
        depth=6,
        verbose=0,
        random_state=42
    ),

    "XGBoost": XGBClassifier(
        n_estimators=300,
        learning_rate=0.05,
        max_depth=6,
        subsample=0.8,
        colsample_bytree=0.8,
        eval_metric="logloss",
        random_state=42,
        n_jobs=-1
    ),

    "LightGBM": LGBMClassifier(
        n_estimators=300,
        learning_rate=0.05,
        num_leaves=31,
        random_state=42,
        verbosity=-1
    )
}


# ==========================
# SAMPLERS
# ==========================

samplers = {

    "No Sampling": None,

    "SMOTE": SMOTE(
        sampling_strategy=0.5,
        random_state=42
    ),

    "SMOTE_Tomek": SMOTETomek(
        sampling_strategy=0.5,
        random_state=42
    ),

    "ADASYN": ADASYN(
        sampling_strategy=0.5,
        random_state=42
    )
}


# ==========================
# CV
# ==========================

cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)


results=[]


# ==========================
# TRAIN LOOP
# ==========================

for sampler_name, sampler in samplers.items():

    for model_name, model in models.items():


        print(
            f"Running {sampler_name} + {model_name}"
        )


        if sampler is not None:

            pipeline = Pipeline([

                ("sampler", sampler),

                ("model", model)

            ])

        else:

            pipeline = model


        probs = cross_val_predict(
            pipeline,
            X,
            y,
            cv=cv,
            method="predict_proba",
            n_jobs=-1
        )[:,1]


        preds = (
            probs >= 0.5
        ).astype(int)


        results.append({

            "Sampling":
                sampler_name,

            "Model":
                model_name,

            "Accuracy":
                accuracy_score(
                    y,
                    preds
                ),

            "ROC-AUC":
                roc_auc_score(
                    y,
                    probs
                ),

            "PR-AUC":
                average_precision_score(
                    y,
                    probs
                ),

            "Precision":
                precision_score(
                    y,
                    preds
                ),

            "Recall":
                recall_score(
                    y,
                    preds
                ),

            "F1":
                f1_score(
                    y,
                    preds
                )

        })


final_sampling_results = (
    pd.DataFrame(results)
    .sort_values(
        "PR-AUC",
        ascending=False
    )
    .reset_index(drop=True)
)


display(final_sampling_results)

In [ ]:
import optuna

from catboost import CatBoostClassifier
from sklearn.model_selection import StratifiedKFold, cross_val_score


X = df3.drop(columns=["SeriousDlqin2yrs"])
y = df3["SeriousDlqin2yrs"]


cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)


def objective(trial):

    params = {

        "iterations": trial.suggest_int(
            "iterations",
            300,
            1000
        ),

        "learning_rate": trial.suggest_float(
            "learning_rate",
            0.01,
            0.1,
            log=True
        ),

        "depth": trial.suggest_int(
            "depth",
            4,
            10
        ),

        "l2_leaf_reg": trial.suggest_float(
            "l2_leaf_reg",
            1,
            10
        ),

        "random_strength": trial.suggest_float(
            "random_strength",
            0,
            2
        ),

        "bagging_temperature": trial.suggest_float(
            "bagging_temperature",
            0,
            5
        ),

        "border_count": trial.suggest_int(
            "border_count",
            32,
            255
        ),

        "loss_function":
            "Logloss",

        "eval_metric":
            "AUC",

        "auto_class_weights":
            "Balanced",

        "random_seed":
            42,

        "verbose":
            0
    }


    model = CatBoostClassifier(
        **params
    )


    score = cross_val_score(
        model,
        X,
        y,
        cv=cv,
        scoring="roc_auc",
        n_jobs=-1
    ).mean()


    return score



study = optuna.create_study(
    direction="maximize"
)


study.optimize(
    objective,
    n_trials=100
)


print(
    study.best_value
)


print(
    study.best_params
)

In [ ]:
best_params = study.best_params


final_catboost = CatBoostClassifier(

    **best_params,

    loss_function="Logloss",

    eval_metric="AUC",

    auto_class_weights="Balanced",

    random_seed=42,

    verbose=0
)


final_catboost.fit(
    X,
    y
)

In [ ]:
probs = final_catboost.predict_proba(
    X
)[:,1]

In [ ]:
import numpy as np
import pandas as pd

from sklearn.metrics import (
    precision_score,
    recall_score,
    f1_score
)


threshold_results=[]


for threshold in np.arange(
    0.01,
    0.99,
    0.01
):

    preds = (
        probs >= threshold
    ).astype(int)


    recall = recall_score(
        y,
        preds
    )


    precision = precision_score(
        y,
        preds
    )


    f1 = f1_score(
        y,
        preds
    )


    if recall >= 0.75:

        threshold_results.append({

            "Threshold":
                threshold,

            "Precision":
                precision,

            "Recall":
                recall,

            "F1":
                f1
        })


threshold_df = pd.DataFrame(
    threshold_results
)


threshold_df.sort_values(
    "F1",
    ascending=False
).head(10)

In [ ]:
best_threshold = (
    threshold_df
    .sort_values(
        "F1",
        ascending=False
    )
    .iloc[0]["Threshold"]
)


final_preds = (
    probs >= best_threshold
).astype(int)


from sklearn.metrics import classification_report


print(
    "Threshold:",
    best_threshold
)


print(
    classification_report(
        y,
        final_preds
    )
)

In [ ]:
from sklearn.metrics import confusion_matrix

confusion_matrix(
    y,
    final_preds
)

# Final Credit Risk Model Results

## Final Model

Dataset:
- df3

Model:
- CatBoost Classifier

Class Imbalance Handling:
- auto_class_weights="Balanced"

Hyperparameter Optimization:
- Optuna

Threshold Optimization:
- Yes

Final Classification Threshold:



---

# Final Performance

| Metric | Score |
|---|---:|
| Accuracy | 83% |
| Precision | 24.59% |
| Recall | 75.11% |
| F1-Score | 37.05% |

---




In [ ]:
import shap
import matplotlib.pyplot as plt
import pandas as pd

In [ ]:
explainer = shap.TreeExplainer(
    final_catboost
)

In [ ]:
shap_values = explainer.shap_values(
    X
)

In [ ]:
shap_values.shape

In [ ]:
shap.summary_plot(
    shap_values,
    X,
    plot_type="bar"
)

In [ ]:
shap.summary_plot(
    shap_values,
    X
)

In [ ]:
shap.dependence_plot(
    "RevolvingUtilization_log",
    shap_values,
    X
)

In [ ]:
shap.dependence_plot(
    "DelinquencySeverityScore",
    shap_values,
    X
)

In [ ]:
shap.dependence_plot(
    "TotalLatePayments",
    shap_values,
    X
)

In [ ]:
probs = final_catboost.predict_proba(X)[:,1]


high_risk_index = probs.argmax()


customer = X.iloc[high_risk_index]

In [ ]:
customer_shap = shap_values[high_risk_index]

In [ ]:
shap.waterfall_plot(
    shap.Explanation(
        values=customer_shap,
        base_values=explainer.expected_value,
        data=customer.values,
        feature_names=X.columns
    )
)